In [1]:

import osprey
osprey.start()

OSPREY 3.3-dev, Python 3.10.16, Java 17.0.15, Linux-5.15.0-140-generic-x86_64-with-glibc2.35
Using up to 1024 MiB heap memory: 128 MiB for garbage, 896 MiB for storage


In [2]:

# choose a forcefield
ffparams = osprey.ForcefieldParams()

In [7]:
# read a PDB file for molecular info
mol = osprey.readPdb('/home/users/ys472/ying_Project/kinase_inhibitor_design/pkn2.cleaned.filtered.pdb')

read PDB file from file: /home/users/ys472/ying_Project/kinase_inhibitor_design/pkn2.cleaned.filtered.pdb


In [8]:

# make sure all strands share the same template library
templateLib = osprey.TemplateLibrary(ffparams.forcefld)

In [9]:
templateLib

<java object 'edu.duke.cs.osprey.restypes.ResidueTemplateLibrary'>

In [15]:
# define the protein strand
protein = osprey.Strand(mol, templateLib=templateLib, residues=['A1', 'A334'])

#Should I keep the protein regid?
#protein.flexibility['G649'].setLibraryRotamers(osprey.WILD_TYPE, 'TYR', 'ALA', 'VAL', 'ILE', 'LEU').addWildTypeRotamers().setContinuous()
#protein.flexibility['G650'].setLibraryRotamers(osprey.WILD_TYPE).addWildTypeRotamers().setContinuous()
#protein.flexibility['G651'].setLibraryRotamers(osprey.WILD_TYPE).addWildTypeRotamers().setContinuous()
#protein.flexibility['G654'].setLibraryRotamers(osprey.WILD_TYPE).addWildTypeRotamers().setContinuous()

ARG A 321


In [17]:
# Define the ligand strand
ligand = osprey.Strand(mol, templateLib=templateLib, residues=['B1', 'B14'])

# Add flexibility and potential mutations to key ligand residues
ligand.flexibility['B2'].setLibraryRotamers(osprey.WILD_TYPE, 'PHE', 'TYR').addWildTypeRotamers().setContinuous()
ligand.flexibility['B5'].setLibraryRotamers(osprey.WILD_TYPE, 'LEU', 'ILE', 'VAL').addWildTypeRotamers().setContinuous()
ligand.flexibility['B8'].setLibraryRotamers(osprey.WILD_TYPE, 'ASP', 'GLU').addWildTypeRotamers().setContinuous()
ligand.flexibility['B10'].setLibraryRotamers(osprey.WILD_TYPE).addWildTypeRotamers().setContinuous()
ligand.flexibility['B12'].setLibraryRotamers(osprey.WILD_TYPE, 'ASN', 'GLN').addWildTypeRotamers().setContinuous()
ligand.flexibility['B14'].setLibraryRotamers(osprey.WILD_TYPE, 'SER', 'THR').addWildTypeRotamers().setContinuous()


<java object 'edu.duke.cs.osprey.confspace.Strand.ResidueFlex'>

In [18]:
# make the conf space for the protein
proteinConfSpace = osprey.ConfSpace(protein)

In [19]:
# make the conf space for the ligand
ligandConfSpace = osprey.ConfSpace(ligand)

In [20]:
# make the conf space for the protein+ligand complex
complexConfSpace = osprey.ConfSpace([protein, ligand])


In [21]:
# how should we compute energies of molecules?
# (give the complex conf space to the ecalc since it knows about all the templates and degrees of freedom)
parallelism = osprey.Parallelism(cpuCores=4)
ecalc = osprey.EnergyCalculator(complexConfSpace, ffparams, parallelism=parallelism)

In [22]:
# configure K*
kstar = osprey.KStar(
	proteinConfSpace,
	ligandConfSpace,
	complexConfSpace,
	epsilon=0.99, # you proabably want something more precise in your real designs
	writeSequencesToConsole=True,
	writeSequencesToFile='kstar.results.tsv'
)

In [23]:
# configure K* inputs for each conf space
for info in kstar.confSpaceInfos():

	# how should we define energies of conformations?
	eref = osprey.ReferenceEnergies(info.confSpace, ecalc)
	info.confEcalc = osprey.ConfEnergyCalculator(info.confSpace, ecalc, referenceEnergies=eref)

	# compute the energy matrix
	emat = osprey.EnergyMatrix(info.confEcalc, cacheFile='emat.%s.dat' % info.id)

	# how should we score each sequence?
	# (since we're in a loop, need capture variables above by using defaulted arguments)
	def makePfunc(rcs, confEcalc=info.confEcalc, emat=emat):
		return osprey.PartitionFunction(
			confEcalc,
			osprey.AStarTraditional(emat, rcs, showProgress=False),
			osprey.AStarTraditional(emat, rcs, showProgress=False),
			rcs
		)
	info.pfuncFactory = osprey.KStar.PfuncFactory(makePfunc)


Calculating reference energies for 0 residue confs...
Calculating energy matrix with 0 entries
wrote energy matrix to file: /home/users/ys472/ying_Project/kinase_inhibitor_design/emat.protein.dat
Calculating reference energies for 198 residue confs...
Progress:  100.0%   ETA: 0 ns
Finished in 278.0 ms
Calculating energy matrix with 15289 entries
Progress:   41.3%   ETA: 8.0 s
Progress:   92.3%   ETA: 1.9 s
Progress:  100.0%   ETA: 772.0 ms
Finished in 10.7 s
wrote energy matrix to file: /home/users/ys472/ying_Project/kinase_inhibitor_design/emat.ligand.dat
Calculating reference energies for 198 residue confs...
Progress:  100.0%   ETA: 0 ns
Finished in 267.2 ms
Calculating energy matrix with 15289 entries
Progress:   14.8%   ETA: 30.2 s
Progress:   27.3%   ETA: 31.1 s
Progress:   35.5%   ETA: 28.1 s
Progress:   45.4%   ETA: 26.0 s
Progress:   59.5%   ETA: 20.0 s
Progress:   68.9%   ETA: 14.6 s
Progress:   77.1%   ETA: 10.3 s
Progress:   94.0%   ETA: 4.7 s
Progress:  100.0%   ETA: 1.6 s

In [24]:
# run K*
scoredSequences = kstar.run(ecalc.tasks)

computing K* scores for 12 sequences to epsilon = 0.99 ...
sequence    1/  12   B2=pro B5=arg B8=lys B12=leu B14=lys   K*(log10): none      in [-Infinity,-Infinity] (log10)   protein: [0.000000 , 0.000000] (log10)                   , numConfs:    1, delta: 0.000   ligand: [19.091457,21.061306] (log10)                   , numConfs:   27, delta: 0.989   complex: [-Infinity,-Infinity] (log10,OutOfLowEnergies)  , numConfs:    5, delta: NaN
sequence    2/  12   B2=pro B5=arg B8=lys B12=leu B14=SER   K*(log10): none      in [-Infinity,-Infinity] (log10)   protein: [0.000000 , 0.000000] (log10)                   , numConfs:    1, delta: 0.000   ligand: [19.268225,21.248247] (log10)                   , numConfs:   88, delta: 0.990   complex: [-Infinity,-Infinity] (log10,OutOfLowEnergies)  , numConfs:    5, delta: NaN
sequence    3/  12   B2=pro B5=arg B8=lys B12=leu B14=THR   K*(log10): none      in [-Infinity,-Infinity] (log10)   protein: [0.000000 , 0.000000] (log10)                   , numC

In [25]:
# make a sequence analyzer to look at the results
analyzer = osprey.SequenceAnalyzer(kstar)


In [26]:
# use results
for scoredSequence in scoredSequences:
	print("result:")
	print("\tsequence: %s" % scoredSequence.sequence)
	print("\tK* score: %s" % scoredSequence.score)

	# write the sequence ensemble, with up to 10 of the lowest-energy conformations
	numConfs = 10
	analysis = analyzer.analyze(scoredSequence.sequence, numConfs)
	print(analysis)
	analysis.writePdb(
		'seq.%s.pdb' % scoredSequence.sequence,
		'Top %d conformations for sequence %s' % (numConfs, scoredSequence.sequence)
	)

result:
	sequence: B2=pro B5=arg B8=lys B12=leu B14=lys
	K* score: none      in [-Infinity,-Infinity] (log10)
Residues           B2    B5    B8    B10   B12   B14  complex Sequence   pro   arg   lys   leu   lys  
Ensemble of 0 conformations:

result:
	sequence: B2=pro B5=arg B8=lys B12=leu B14=SER
	K* score: none      in [-Infinity,-Infinity] (log10)
Residues           B2    B5    B8    B10   B12   B14  complex Sequence   pro   arg   lys   leu   SER  
Ensemble of 0 conformations:

result:
	sequence: B2=pro B5=arg B8=lys B12=leu B14=THR
	K* score: none      in [-Infinity,-Infinity] (log10)
Residues           B2    B5    B8    B10   B12   B14  complex Sequence   pro   arg   lys   leu   THR  
Ensemble of 0 conformations:

result:
	sequence: B2=pro B5=arg B8=lys B12=ASN B14=lys
	K* score: none      in [-Infinity,-Infinity] (log10)
Residues           B2    B5    B8    B10   B12   B14  complex Sequence   pro   arg   lys   ASN   lys  
Ensemble of 0 conformations:

result:
	sequence: B2=pro B5